## Wikidata linked data

# Wikidata Linked Data Enrichment & Mapping

This notebook enriches the project's location data by fetching detailed information from Wikidata and generating a visualization map.

**Workflow:**
1.  **Collect IDs**: Scans CSV files in `inferences/wikidata-references/` to gather all unique Wikidata IDs linked in previous analysis steps.
2.  **Fetch Data**: Connects to the **Wikidata API** (using `pywikibot`) to retrieve metadata for each location, including:
    *   **Labels & Descriptions**: In English, Portuguese, and Chinese.
    *   **Coordinates**: Latitude and Longitude.
    *   **Hierarchy**: Administrative territories and Countries.
3.  **Cache Management**: checks a local cache (`locations_wikidata_info.xlsx`) to avoid redundant API calls, updating it only with new entries.
4.  **Visualize**: Generates an interactive map (`map_all_locations.html`) using Plotly to display all geolocated points.

In [ ]:
import glob
import pandas as pd

# get list of all csv files in directory `../inferences/wikidata-references/`
csv_files = glob.glob("../inferences/wikidata-references/*.csv")
wikidata_entities = set()
for file in csv_files:
    print(file)
    df = pd.read_csv(file)
    # Assuming the wikidata id column is named 'wikidata_id'
    if 'wikidata_id' in df.columns:
        wikidata_entities.update(df['wikidata_id'].dropna().astype(str))
    print(len(df))
wikidata_entities = list(set(wikidata_entities))
print(len(wikidata_entities))


## Fetch wikidata data

In [ ]:
%pip install pywikibot

Get all the wikidata ids in the database


In [ ]:
from dehergne_util import locations_wikidata_info_file

# preload dataframe entities_df from xls file "../inferences/locations_wikidata_info.xlsx"
entities_cache = pd.read_excel(locations_wikidata_info_file, index_col=0)


In [ ]:
entities_cache.loc['Q1207099',]


In [ ]:
entities_cache.sample(10)

In [ ]:
import time
import pywikibot
import pandas as pd
from dehergne_util import locations_wikidata_info_file

site = pywikibot.Site("wikidata", "wikidata")
repo = site.data_repository()
use_cache = True

entities_data = []
entities_from_cache = []
problems = []  # to store any problems encountered during the fetching process
for wikidata_id in wikidata_entities:
    if wikidata_id == 'Q11146687':
        pass
    # check if the wikidata_id is already in the cache
    if use_cache and wikidata_id in entities_cache.index:
        entities_from_cache.append(wikidata_id)
        # print(f"{wikidata_id} already in cache, skipping...")
        continue
    entity_data = {}
    # Fetch the item from Wikidata
    entity_data['wikidata_id'] = wikidata_id
    print(f"Fetching {wikidata_id} from Wikidata...")
    # Use the repository to get the item
    try:
        item = pywikibot.ItemPage(repo, wikidata_id)
    except Exception as e:
        problems.append({'wikidata_id': wikidata_id, 'exception': str(e)})
        continue
    item_dict = item.get()
    entity_data['chinese_label'] = item_dict['labels'].get('zh', None)
    entity_data['english_label'] = item_dict['labels'].get('en', None)
    entity_data['portuguese_label'] = item_dict['labels'].get('pt', None)
    entity_data['english_description'] = item_dict['descriptions'].get('en', None)
    entity_data['chinese_description'] = item_dict['descriptions'].get('zh', None)
    entity_data['portuguese_description'] = item_dict['descriptions'].get('pt', None)
    # get the coordinates if they exist
    coord = None
    if 'P625' in item_dict['claims']:
        coord_claim = item_dict['claims']['P625'][0].getTarget()
        coord = (coord_claim.lat, coord_claim.lon)
    entity_data['coordinates'] = coord
    # store lat and lon in separate keys
    entity_data['latitude'] = coord[0] if coord else None
    entity_data['longitude'] = coord[1] if coord else None
    # get the administrative territorial entity if it exists (P131)
    administrative_entity = None
    administrative_entity_label_en = None
    administrative_entity_label_zh = None
    administrative_entity_id = None
    administrative_entity_claims = item_dict['claims'].get('P131', [])
    if administrative_entity_claims:
        administrative_entity = administrative_entity_claims[0].getTarget()
        administrative_entity_id = administrative_entity.id
        
        # Fetch data once to be efficient
        admin_data = administrative_entity.get()
        
        administrative_entity_label_en = admin_data['labels'].get('en', None)
        administrative_entity_label_zh = admin_data['labels'].get('zh', None)
        type_claims = admin_data['claims'].get('P31', [])
        if type_claims:
            # Get the first 'instance of' value (e.g., "Province")
            type_item = type_claims[0].getTarget()
            try:
                # Fetch the label of the type
                administrative_entity_type = type_item.get()['labels'].get('en', None)
            except Exception:
                pass
    # store administrative information in entity_data
    entity_data['administrative_entity_id'] = administrative_entity_id
    entity_data['administrative_entity_label_en'] = administrative_entity_label_en
    entity_data['administrative_entity_label_zh'] = administrative_entity_label_zh
    entity_data['administrative_entity_type'] = administrative_entity_type # <--- Store it

    # get the country
    country = None
    country_label = None
    country_id = None
    country_claims = item_dict['claims'].get('P17', [])
    if country_claims:
        country = country_claims[0].getTarget()
        country_id = country.id
        country_label = country.get()['labels'].get('en', None)
    entity_data['country_id'] = country_id
    entity_data['country_label'] = country_label
    # append the entity data to the list
    entities_data.append(entity_data)

    print(f"Fetched {wikidata_id} with labels: {entity_data['english_label']}, {entity_data['portuguese_label']}, {entity_data['chinese_label']}, {entity_data['administrative_entity_label_en']}, {entity_data['administrative_entity_label_zh']}, {entity_data['country_label']}")
    time.sleep(0.5)  # to avoid hitting the API too hard

# Convert the list of dictionaries to a DataFrame
entities_df = pd.DataFrame(entities_data)
if len(entities_df) == 0:
    print("No new entities found.")
    entities_df = entities_cache  # use the cached data if no new entities found
else:
    print(f"Fetched {len(entities_df)} new entities from Wikidata.")
    entities_df.set_index('wikidata_id', inplace=True)

# get the rows entities_cache in entities_from_cache list of ids
# this removes from cache entities not in the current run
entities_cache = entities_cache[entities_cache.index.isin(entities_from_cache)]
# append the new data to the existing cache
entities_df = pd.concat([entities_cache, entities_df], axis=0)



In [ ]:
# print the problems encountered one per line
for problem in problems:
    print(f"Problem with {problem['wikidata_id']}: {problem['exception']}")

In [ ]:
# show duplicates in the entities_df use index
duplicates = entities_df.index[entities_df.index.duplicated()].unique()
duplicates

In [ ]:
# remove duplicates from entities_df
print(f"Removing {len(duplicates)} duplicates from entities_df, before removal: {len(entities_df)} rows.")
entities_df = entities_df[~entities_df.index.duplicated(keep='first')]
print(f"After removal: {len(entities_df)} rows.")

In [ ]:
entities_df.info()
entities_df.head(10)

In [ ]:
entities_df.loc['Q1207099',]

In [ ]:
from dehergne_util import locations_wikidata_info_file
entities_df.to_excel(locations_wikidata_info_file, index=True)
print(f"Saved {len(entities_df)} entities to {locations_wikidata_info_file}")

Entries with no coordinates

Ensure we have plotly and nbformat. Restart the kernel if necessary.

In [ ]:
%pip install plotly nbformat

In [ ]:
import plotly.express as px
import plotly.io as pio

# Create a 'label' column by concatenating 'english_label' and 'chinese_label'
# Handle potential None values by replacing them with empty strings
entities_df['label'] = entities_df['english_label'].fillna('') + ' / ' + entities_df['chinese_label'].fillna('')

# Drop rows where latitude or longitude is NaN before plotting
entities_df_map = entities_df.dropna(subset=['latitude', 'longitude'])

fig = px.scatter_map(
    entities_df_map,
    lat="latitude",
    lon="longitude",
    hover_name="label",  # Use the new 'label' column for hover text
    text="label",        # Show the label on the map
    zoom=3,

)

fig.update_layout(mapbox_style="open-street-map")
fig.update_layout(autosize=True)
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
#fig.update_traces(textfont=dict(size=6))
fig.show(config={"responsive": True})

# Export the figure to an HTML file
pio.write_html(fig,
                config={"responsive": True},
                include_plotlyjs=True,
                file='../inferences/map_all_locations.html',
                auto_open=True)